# 환경 설정

In [ ]:
!pip install -U langchain langchain-core langchain-google-genai cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 988.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Atte

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
# 코랩 Secrets에서 GOOGLE_API_KEY를 가져오기

google_api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = google_api_key


In [ ]:
import cohere
api_key = userdata.get('COHERE_API_KEY')
co = cohere.Client(api_key)

# 데이터

## 소설 원문과 요약문 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 저장된 소설 데이터 불러오기
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/[파일 경로]/novel_summary.csv")
stored_title = list(df['title'])
stored_content = list(df['contents'])
stored_summary = list(df['summary'])
print(stored_title[0])
print(stored_content[0][:50])
print(stored_summary[0][:50])

백치(白痴) 아다다 1화
질그릇이 땅에 부딪치는 소리가 났다고 들렸는데, 마당에는 아무도 없다.
부엌에 쥐가 들었나
안녕하세요! 요청하신 소설 **<백치(白痴) 아다다> 1화**의 내용을 친절하게 요약해 드


# 하이브리드 검색

## 의미 검색

In [ ]:
from sentence_transformers import SentenceTransformer
#sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
#sentence_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
sentence_model = SentenceTransformer('intfloat/multilingual-e5-base')
embeddings_s = sentence_model.encode(["passage: " + c for c in stored_summary]) #요약문 임베딩
embeddings_s.shape

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

(97, 768)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def dense_vector_search(query, contents, embeddings, k):
    emd = sentence_model.encode("query: " + query) #임베딩
    sim_scores = [cosine_similarity([embeddings[i]], [emd]) for i in range(len(contents))]
    index = range(0, len(contents))
    pairs = zip(sim_scores, index)
    result = sorted(pairs, reverse=True)[:k]
    return zip(*result)


## 키워드 검색

In [ ]:
import math
import numpy as np
from typing import List
from transformers import PreTrainedTokenizer, AutoTokenizer
from collections import defaultdict

class BM25:
  def __init__(self, corpus:List[List[str]], tokenizer:PreTrainedTokenizer):
    self.tokenizer = tokenizer
    self.corpus = corpus
    self.tokenized_corpus = self.tokenizer(corpus, add_special_tokens=False)['input_ids']
    self.n_docs = len(self.tokenized_corpus)
    self.avg_doc_lens = sum(len(lst) for lst in self.tokenized_corpus) / len(self.tokenized_corpus)
    self.idf = self._calculate_idf()
    self.term_freqs = self._calculate_term_freqs()

  def _calculate_idf(self):
    idf = defaultdict(float)
    for doc in self.tokenized_corpus:
      for token_id in set(doc):
        idf[token_id] += 1
    for token_id, doc_frequency in idf.items():
      idf[token_id] = math.log(((self.n_docs - doc_frequency + 0.5) / (doc_frequency + 0.5)) + 1)
    return idf

  def _calculate_term_freqs(self):
    term_freqs = [defaultdict(int) for _ in range(self.n_docs)]
    for i, doc in enumerate(self.tokenized_corpus):
      for token_id in doc:
        term_freqs[i][token_id] += 1
    return term_freqs

  def get_scores(self, query:str, k1:float = 1.2, b:float=0.75):
    query = self.tokenizer([query], add_special_tokens=False)['input_ids'][0]
    scores = np.zeros(self.n_docs)
    for q in query:
      idf = self.idf[q]
      for i, term_freq in enumerate(self.term_freqs):
        q_frequency = term_freq[q]
        doc_len = len(self.tokenized_corpus[i])
        score_q = idf * (q_frequency * (k1 + 1)) / ((q_frequency) + k1 * (1 - b + b * (doc_len / self.avg_doc_lens)))
        scores[i] += score_q
    return scores

  def get_top_k(self, query:str, k:int):
    scores = self.get_scores(query)
    top_k_indices = np.argsort(scores)[-k:][::-1]
    top_k_scores = scores[top_k_indices]
    return top_k_scores, top_k_indices

b_tokenizer = AutoTokenizer.from_pretrained('klue/roberta-base')
bm25 = BM25(stored_content, b_tokenizer)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1567 > 512). Running this sequence through the model will result in indexing errors


## 하이브리드 검색

In [ ]:
##비율 합산
def new_rank(scores:List[List[int]], rankings):
    rrf = defaultdict(float)
    for i in range(len(scores[0])):
            rrf[rankings[0][i]] +=  scores[0][i] * 0.1
            rrf[rankings[1][i]] +=  scores[1][i] * 0.9
            #print(i, s)
    return sorted(rrf, key=rrf.get, reverse=True)

In [ ]:
## 하이브리드 검색
def hybrid_search(query, contents, embeddings, bm25):
  d_scores, dense_search_ranking = dense_vector_search(query, contents, embeddings, 30)#의미 검색
  b_scores, bm25_search_ranking = bm25.get_top_k(query, 30) #bm25 (키워드) 검색
  for i in range(len(b_scores)):
    b_scores[i] /=  (10 + b_scores[i])
  results = []
  results = new_rank(scores = [d_scores, b_scores], rankings=[dense_search_ranking, bm25_search_ranking]) #순위 조합
  return results

## 재순위화

In [ ]:
def rerank(query: str, ids: list, top_k: int = 3) -> list:
    docs = [stored_content[i] for i in ids] #docs는 인덱스 리스트

    response = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=docs,
    top_n=top_k  # 상위 3개만 반환
    )
    results = response.results
    return [ids[x.index] for x in results]

# rag

In [ ]:
## 대화 요약본 생성
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sqlite3

a = 0
model = ChatGoogleGenerativeAI(model="gemma-4-26b-a4b-it")
novel_data = [stored_title[i]+'\n'+stored_content[i] for i in range(len(stored_content))]
print(novel_data[0][:40])
chat_summary = ""

for i, c in enumerate(novel_data):
    summary_prompt = ChatPromptTemplate.from_template(
        """
        다음은 대화 요약본과 최근 대화 입니다.

        #대화 요약본: {chat_summary}

        #최근 대화: {recent_conversation}

        위 내용을 바탕으로 아래 항목을 포함하여 간결하게 요약하세요. (3000자 이내)
        -논의된 소설들의 제목 및 전반적인 특징, 작풍
        -주요 등장인물과 관계
        -중요하게 언급된 장면이나 주제
        """
    )
    summary_chain = summary_prompt | model | StrOutputParser()
    recent_conversation = {'user': c, 'ai': stored_summary[i]}
    chat_summary = summary_chain.invoke({'recent_conversation': recent_conversation, "chat_summary": chat_summary})
print(chat_summary)

백치(白痴) 아다다 1화
질그릇이 땅에 부딪치는 소리가 났다고 들렸는데,
제시된 대화 요약본과 <인간문제> 14화의 내용을 통합하여 정리한 최종 요약본입니다.

---

### **1. 논의된 소설의 제목 및 전반적인 특징, 작풍**
* **대상 작품:** <백치 아다다>, <며느리>, <죄와 벌>, <검둥이>, <동정>, <마약>, <번뇌>, <산남>, <소금>, <어둠>, <어머니와 딸>, <원고료 이백원>, **<인간문제>** 등.
* **작풍 (사실주의, Realism):**
    * **사회적 사실주의:** 빈곤, 계급 격차(지주와 소작인), 농촌의 황폐화, 물질 만능주의가 개인의 삶에 미치는 영향을 가감 없이 묘사함.
    * **심리적·도덕적 사실주의:** 극한의 상황에서 나타나는 인간의 이기심, 수치심, 고독, 그리고 고된 노동과 그리움 속에서도 삶을 견뎌내는 인물들의 내면을 심도 있게 탐구함.
    * **대비적 묘사:** 개인의 소소한 행복과 냉혹한 사회적 시선, 인물의 강렬한 내면적 갈망과 비루한 현실을 대비시켜 주제를 부각함.

### **2. 주요 등장인물과 관계**
* **선비:** 김민수의 딸. 고운 외모와 정갈한 성품을 지녔으며, 눈가의 검은 사마귀가 특징임. '첫째'의 애틋한 연모의 대상이자, 방문객 '신철'이 호기심과 매력을 느끼는 대상임.
* **첫째:** 선비를 향한 깊은 연모로 밤잠을 설치는 인물. 어머니의 부적절한 행실(유 서방과의 만남)을 목격하고 깊은 분노와 원망을 느낌.
* **신철:** 옥점과 함께 덕호의 집에 머무는 인물. 옥점과 장래 부부로 여겨지나, 실제로는 선비의 정갈한 모습에 매료되어 그녀를 만나기 위해 옥점을 따돌리고 선비를 쫓음.
* **옥점:** 덕호의 딸. 신철과 함께 시간을 보내며 그를 남성적으로 느끼지만, 신철의 시선이 선비에게 향해 있음을 알지 못함.
* **첫째의 어머니:** 유 서방과 은밀한 관계를 맺고 있으며, 이로 인해 아들인 첫째에게 원망을 사는 인물.
* **정덕호:** 물질 

In [ ]:
##rag 이용#
###저장 하는 부분 코드 수정하기~~ 아직 안 함
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sqlite3

model = ChatGoogleGenerativeAI(model="gemma-4-26b-a4b-it")
def chatbot():
  print("\n--- 챗봇 시작 ---")
  print("종료하려면 'exit' 또는 'quit'을 입력하세요.")
  print("챗봇: 안녕하세요. 무엇을 도와드릴까요?")

  global chat_summary #생성한 대화 요약본 사용

  answer = ''
  while True:
    user_input = input("입력: ")
    if user_input.lower() in ["exit", "quit"]:
        print("챗봇을 종료합니다.")
        break

    print('\n\n\n-----답변 시작-----\n\n')

    #검색
    candidates = hybrid_search(user_input, stored_content, embeddings_s, bm25)[:15]
    results = rerank(user_input, candidates, 3)

    novel_content = stored_content[results[0]]
    novel_content2 = stored_content[results[1]]
    novel_content3 = stored_content[results[2]]
    print(f'뽑힌 소설: {stored_title[results[0]], stored_title[results[1]], stored_title[results[2]]}\n\n')

    prompt = ChatPromptTemplate.from_template(
    """

      당신은 친절한 한국어 챗봇입니다.

      답변 규칙:
      1. 질문이 소설의 특정 내용(인물, 장면, 대사 등)에 관한 것이면 [참고할 소설 내용]을 참고하여 답변하세요.
      -참고한 부분을 원문으로 알려주세요.
      -소설 내용에 없는 내용은 지어내지 마세요.
      2. 질문이 소설 전반의 감상, 분석이나 작가 의도에 관한 것이면 [대화 요약]을 참고하여 답변하세요.
      3. 소설과 무관한 질문이면 자유롭게 답변하세요.

      [참고할 소설 내용]
      #1: {novel_content}
      #2: {novel_content2}
      #3: {novel_content3}
      [대화 요약]
      {chat_summary}

      #질문: {user_input}
    """
    )


    chain = prompt | model | StrOutputParser()
    answer = chain.invoke({'novel_content': novel_content, 'novel_content2':novel_content2, 'novel_content3':novel_content3, 'chat_summary':chat_summary, 'user_input': user_input})

    print(f"\nai 답변: {answer}\n\n")


    #대화 후 요약 실행
    summary_prompt = ChatPromptTemplate.from_template(
        """
        다음은 대화 요약본과 최근 대화 입니다.

        #대화 요약본: {chat_summary}

        #최근 대화: {recent_conversation}

        위 내용을 바탕으로 아래 항목을 중심으로 간결하게 요약하세요. (1000자 이내)
        -논의된 소설들의 제목 및 전반적인 특징, 작풍
        -주요 등장인물과 관계
        -중요하게 언급된 장면이나 주제
        """
    )
    summary_chain = summary_prompt | model | StrOutputParser()
    recent_conversation = {'user': user_input, 'ai': answer}
    chat_summary = summary_chain.invoke({'recent_conversation': recent_conversation, "chat_summary": chat_summary})
    print('--------------------\n')

In [ ]:

# 아다다가 '아다다'라고 불리게 된 이유는? #백치 아다다
# K선생이 생각한, 스스로의 단점은? #검둥이
# 지금까지 읽은 소설들 기억해?
# 산남에서, 조수가 데려온 사나이는 누구인가요?
# 미세먼지가 뭐야?
# 시험에 합격한 옥이가 떠올린 이는? #어머니와 딸

chatbot()


--- 챗봇 시작 ---
종료하려면 'exit' 또는 'quit'을 입력하세요.
챗봇: 안녕하세요. 무엇을 도와드릴까요?
입력: 아다다가 '아다다'라고 불리게 된 이유는?



-----답변 시작-----


뽑힌 소설: ('백치(白痴) 아다다 1화', '백치(白痴) 아다다 2화', '백치(白痴) 아다다 3화')



ai 답변: 아다다가 '아다다'라고 불리게 된 이유는 그녀가 말을 하지 못하는 벙어리이기 때문입니다. 말을 하려고 할 때마다 "아다다"라는 소리만 반복해서 내뱉게 되는데, 사람들이 이를 조롱하기 위해 부르던 것이 이름처럼 굳어진 것입니다. 원래 '확실'이라는 뚜렷한 이름이 있었음에도 불구하고 모두가 그녀를 '아다다'라고 불렀다고 합니다.

**[참고한 원문]**
"아다다는 벙어리였던 것이다. 말을 하렬 때에는 한다는 것이 아다다 소리만이 연거푸 나왔다. 그래서, 이것을 조롱 삼아 확실이라는 뚜렷한 이름이 있음에도 불구하고, 누구나 그를 부르는 이름은 아다다였다."


--------------------

입력: K선생이 생각한, 스스로의 단점은?



-----답변 시작-----


뽑힌 소설: ('검둥이 2화', '검둥이 3화', '검둥이 1화')



ai 답변: K선생이 생각하는 스스로의 가장 큰 단점은 **거짓말을 못 하는 것**과 **한번 옳다고 믿으면 끝까지 믿으려는 것**입니다. 그는 이러한 성격이 사회생활(처세)을 하는 데 있어 가장 큰 단점이라고 생각하고 있습니다.

**[참고한 원문]**
> "억지로 들자면 거짓말 못하는 것, 한번 옳다고 보면 끝까지 믿으려는 것, 이것이 처세로는 가장 큰 단점이 아닐 수가 없다."


--------------------

입력: 지금까지 읽은 소설들 기억해?



-----답변 시작-----


뽑힌 소설: ('번뇌 4화', '원고료 이백원 1화', '죄와벌 3화')



ai 답변: 네, 그럼요! 지금까지 보내주신 소설들의 내용을 아주 잘 기억하고 있습니다. 

보내주신 내용들은 주로 